## Overview

This notebook conducts an in depth study into the use of physics informed neural networks (PINNs) for the 1D heat equation, and tries to quantify how relevant they actually are compared to a classical baseline solver. The main question is whether a PINN is actually worth the extra training and tuning cost when a classical method already solves this problem well.

We look at two versions of the problem.

Forward problem: we know the diffusivity and just need to solve for the temperature. We compare a baseline PINN, using standard hyperparameters from the literature, against a PINN tuned with Optuna (a 300 trial search), and both against the finite difference solver. We also test how the results change across different diffusivity values, and check how the finite difference solver's accuracy improves as the grid gets finer.

Inverse problem: we only have a few noisy temperature measurements and need to recover the diffusivity from them. We compare a baseline PINN, a tuned PINN, and a classical least squares method that uses the finite difference solver (CN-NLS). It would be easy to accidentally evaluate a config on the same data used to pick it, so this is split into three stages: search, confirmation, and final evaluation. That way the number we report is never influenced by the data used to select the winning config. We also test how results change with different noise levels, number of observations, and diffusivity values.

For every comparison we report both accuracy and cost: training time, how long it takes to evaluate the trained model, and how long the Optuna search itself took. We also added a CPU only timing run for the PINNs so training time can be compared on the same hardware as the finite difference solver, since PINN training normally runs on GPU and the finite difference solver runs on CPU.

All of the actual model and training code lives in pinn_core.py. This notebook is what runs it. heat_pinn_basic.ipynb was an earlier version of this same code and has been removed since everything it did is now handled here.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time
import joblib
import optuna
import optuna.visualization as vis

from pinn_core import (
    set_seed, pinn_architecture, sample_points, generate_noisy_data,
    compute_loss, train_forward, compute_loss_inverse, train_inverse,
    fd_solver, cn_nls_baseline,
)

torch.manual_seed(123)
np.random.seed(123)

#for traing_forward and train_inverse: defaults to GPU if available, else CPU.
default_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Auto-detected default device: {default_device}")

Literature informed initial forward config

In [ ]:
forward_config = {
    # N_bc is actually points per boundary (x=0 and x=1), so the total number of boundary points is 2*N_bc.
    "N_f": 10000,
    "N_bc": 200,
    "N_ic": 200,
    "adam_lr": 1e-3,
    "adam_iters": 2000,
    "lbfgs_iters": 500,
    "hidden_size": 20,
    "n_layers": 4,
    "true_alpha": 0.4,
}

**TODO:** Explain why this held-out test grid is built once here and reused by every section below instead of each section building its own (train/test leakage prevention).

In [ ]:
#Test grid for plotting and error computation
x_test = torch.linspace(0, 1, 1000)
t_test = torch.linspace(0, 1, 1000)
X_test, T_test = torch.meshgrid(x_test, t_test, indexing='ij')
x_test_flat = X_test.reshape(-1, 1)
t_test_flat = T_test.reshape(-1, 1)

# Exact solution on the test grid
u_exact_test = torch.sin(torch.pi * x_test_flat) * torch.exp(-forward_config["true_alpha"] * (torch.pi**2) * t_test_flat)

here we train the pinn based off of the literature informed baseline config across 5 different seeds recording training time, rel L2 error, and inference time. 

In [ ]:
#baseline forward PINN from literature informed hyperparameters.
#trained across 5 random seeds to prevent bias from a single lucky seed.
baseline_forward_results = []
for seed in range(5):
    set_seed(seed)
    start = time.time()
    results = train_forward(forward_config, print_training=False)
    results['elapsed'] = time.time() - start
    baseline_forward_results.append(results)

baseline_val_errors = [r['rel_l2'] for r in baseline_forward_results]
baseline_val_times = [r['elapsed'] for r in baseline_forward_results]
print(f"\nBaseline (no tuning):")
print(f"  Error (validation grid): {np.mean(baseline_val_errors):.6e} ± {np.std(baseline_val_errors):.6e}")
print(f"  Time:  {np.mean(baseline_val_times):.2f} ± {np.std(baseline_val_times):.2f} sec")

#Tests the baseline trained on the 5 random seeds on the held-out test grid, and reports the mean and std of the relative L2 error and inference time.
baseline_test_errors = []
baseline_test_linf_errors = []
baseline_infer_times = []
for r in baseline_forward_results:
    model_device = next(r["model"].parameters()).device
    infer_start = time.time()
    with torch.no_grad():
        u_pred_test = r["model"](x_test_flat.to(model_device), t_test_flat.to(model_device)).cpu()
    baseline_infer_times.append(time.time() - infer_start)
    baseline_test_errors.append((torch.norm(u_pred_test - u_exact_test) / torch.norm(u_exact_test)).item())
    baseline_test_linf_errors.append(torch.max(torch.abs(u_pred_test - u_exact_test)).item())

baseline_test_rel_l2_mean = float(np.mean(baseline_test_errors))
baseline_test_rel_l2_std = float(np.std(baseline_test_errors))
baseline_test_linf_mean = float(np.mean(baseline_test_linf_errors))
baseline_test_linf_std = float(np.std(baseline_test_linf_errors))
baseline_train_time_mean = float(np.mean(baseline_val_times))
baseline_train_time_std = float(np.std(baseline_val_times))
baseline_infer_time_mean = float(np.mean(baseline_infer_times))
baseline_infer_time_std = float(np.std(baseline_infer_times))

print(f"  Held-out test rel L2 error: {baseline_test_rel_l2_mean:.6e} ± {baseline_test_rel_l2_std:.6e}")
print(f"  Held-out test L-inf error:  {baseline_test_linf_mean:.6e} ± {baseline_test_linf_std:.6e}")
print(f"  Training time:              {baseline_train_time_mean:.2f} ± {baseline_train_time_std:.2f} sec")
print(f"  Field-evaluation time:      {baseline_infer_time_mean:.4f} ± {baseline_infer_time_std:.4f} sec")

Setup of optuna sweep and search parameters. here we set lambda_pde = 1 and let the other lambda values adjust based off of that normalization. 

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

#Note: lambda_pde is held at one to because all that matters is the relative weighting of the three loss terms, and the other two lambdas are tuned.
FORWARD_FIXED_CONFIG = {
    "lambda_pde": 1.0,
    "true_alpha": 0.4,
}

#search space for Optuna tuning of forward PINN hyperparameters.
#suggest_categorical tests descrete values as shown, suggest_float tests continuous values in the range shown (log=True means the range is sampled logarithmically).
def objective_forward(trial):
    config = {
        **FORWARD_FIXED_CONFIG,
        "hidden_size":  trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
        "n_layers":     trial.suggest_categorical("n_layers", [3, 4, 5]),
        "activation":   trial.suggest_categorical("activation", ["tanh", "sin"]),
        "lambda_bc":    trial.suggest_float("lambda_bc",  0.001, 1000.0, log=True),
        "lambda_ic":    trial.suggest_float("lambda_ic",  0.001, 1000.0, log=True),
        "N_f":          trial.suggest_categorical("N_f", [5000, 10000, 20000]),
        "N_bc":         trial.suggest_categorical("N_bc", [100, 200, 400]),
        "N_ic":         trial.suggest_categorical("N_ic", [100, 200, 400]),
        "adam_lr":      trial.suggest_float("adam_lr", 1e-4, 1e-2, log=True),
        #Smaller amount of optimizer calls to speed up Optuna search, since the goal is to find a good region of hyperparameter space, not to fully converge each trial.
        "adam_iters":   1500,
        "lbfgs_iters":  500,
    }

    results = train_forward(config, print_training = False, trial = trial)
    return results["rel_l2"]

run of optuna search sweep. Running 300 trials with a pruner that prunes based off of the median at intervals of 200 steps after an 8

In [ ]:
if os.path.exists("optuna_forward.db"):
    os.remove("optuna_forward.db")

study = optuna.create_study(
    direction="minimize",
    study_name="forward_pinn_hpo",
    storage="sqlite:///optuna_forward.db",
    load_if_exists=True,
    # Median pruner: cut a trial early if it's doing worse than the median of
    # past trials at the same point. Waits for 15 trials before comparing (need
    # a real median to compare against) and 800 steps into each trial before
    # judging it (early training is noisy), then only rechecks every 200 steps.
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=800, interval_steps=200),
    sampler=optuna.samplers.TPESampler(seed=42)
)

optuna_forward_search_start = time.time()
study.optimize(
    objective_forward,
    n_trials=300,
    show_progress_bar=True
)
optuna_forward_search_time = time.time() - optuna_forward_search_start

joblib.dump(study, 'forward_hpo_study.pkl')

print(f"\nTotal Optuna forward search time: {optuna_forward_search_time:.2f}s")

# Top 5 trials under specific seed (42)
print(f"\nTop 5 trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"{i}. Error: {trial.value:.6e}")
    print(f"   Params: {trial.params}\n")

# Stats
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE] 
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Summary:")
print(f"  Completed: {len(completed)}")
print(f"  Pruned: {len(pruned)}")
print(f"  Pruning saved: {len(pruned)/(len(completed)+len(pruned))*100:.1f}% of trials")

opt_history_fig = vis.plot_optimization_history(study)
opt_history_fig.update_yaxes(type="log")
opt_history_fig.show()
vis.plot_param_importances(study).show()

**TODO:** Explain why the winning config is chosen by mean validation error across 5 seeds rather than any single run's error, and what the representative-run plot two cells below is (and is not) showing.

In [ ]:
#Retrain top 3 configs on a full run of training (3000/1500 adam/lbfgs iterations).
top_configs = [
    {**FORWARD_FIXED_CONFIG, **trial.params, "adam_iters": 3000, "lbfgs_iters": 1500}
    for trial in sorted_trials[:3]
]

all_config_results = []

#Train each of the top 3 configs across 5 random seeds, and report the mean and std of the validation error and training time.
for i, config in enumerate(top_configs, 1):
    config_results = []
    for seed in range(5):
        set_seed(seed)
        start = time.time()
        results = train_forward(config, print_training=False)
        results['elapsed'] = time.time() - start
        config_results.append(results)

    errors = [r['rel_l2'] for r in config_results]
    times = [r['elapsed'] for r in config_results]
    print(f"\nConfig {i}:")
    print(f"  Error (validation grid): {np.mean(errors):.6e} ± {np.std(errors):.6e}")
    print(f"  Time:  {np.mean(times):.2f} ± {np.std(times):.2f} sec")

    all_config_results.append(config_results)

# Pick the winning config by its MEAN validation error across 5 seeds to avoid bias toward a single lucky run.
config_mean_errors = [np.mean([r['rel_l2'] for r in cr]) for cr in all_config_results]
best_config_idx = int(np.argmin(config_mean_errors))
winning_results = all_config_results[best_config_idx]

#Each winning config is evaluated on the held-out test grid for each seed to compute the mean and std of the relative L2 error and inference time. 
#This test grid is never used during HPO or config selection, so this mean/std, not any single run's number, is what should be quoted as "PINN accuracy." 
test_errors = []
test_linf_errors = []
test_infer_times = []
for r in winning_results:
    model_device = next(r["model"].parameters()).device
    infer_start = time.time()
    with torch.no_grad():
        u_pred_test = r["model"](x_test_flat.to(model_device), t_test_flat.to(model_device)).cpu()
    test_infer_times.append(time.time() - infer_start)
    test_errors.append((torch.norm(u_pred_test - u_exact_test) / torch.norm(u_exact_test)).item())
    test_linf_errors.append(torch.max(torch.abs(u_pred_test - u_exact_test)).item())

pinn_test_rel_l2_mean = float(np.mean(test_errors))
pinn_test_rel_l2_std = float(np.std(test_errors))
pinn_test_linf_mean = float(np.mean(test_linf_errors))
pinn_test_linf_std = float(np.std(test_linf_errors))
pinn_infer_time_mean = float(np.mean(test_infer_times))
pinn_infer_time_std = float(np.std(test_infer_times))
pinn_train_time_mean = float(np.mean([r["elapsed"] for r in winning_results]))
pinn_train_time_std = float(np.std([r["elapsed"] for r in winning_results]))

print(f"\nWinning config: Config {best_config_idx + 1}")
print(f"  Held-out test rel L2 error: {pinn_test_rel_l2_mean:.6e} ± {pinn_test_rel_l2_std:.6e}")
print(f"  Held-out test L-inf error:  {pinn_test_linf_mean:.6e} ± {pinn_test_linf_std:.6e}")
print(f"  Training time:              {pinn_train_time_mean:.2f} ± {pinn_train_time_std:.2f} sec")
print(f"  Field-evaluation time:      {pinn_infer_time_mean:.4f} ± {pinn_infer_time_std:.4f} sec")

#plots are based on whichever seed's model is closest to the mean validation error.
representative_idx = int(np.argmin([abs(r['rel_l2'] - config_mean_errors[best_config_idx]) for r in winning_results]))
best_result = winning_results[representative_idx]

model = best_result["model"]
pinn_train_time = best_result["train_time"]

In [ ]:
# Reuse the held-out test grid built above (x_test/t_test/X_test/T_test/...)
# instead of rebuilding an identical one from scratch -- same points, so
# there's no reason to maintain two copies of the same grid construction.
x, t = x_test, t_test
X, T = X_test, T_test
x_flat, t_flat = x_test_flat, t_test_flat

# Move inputs to whichever device the winning model actually trained on
# (GPU by default, or CPU for an explicit CPU control-timing run), and
# bring the prediction back to CPU right after -- everything downstream
# (numpy conversion, plotting, error math) is unchanged.
model_device = next(model.parameters()).device
pinn_infer_start = time.time()
with torch.no_grad():
    u_pred = model(x_flat.to(model_device), t_flat.to(model_device)).cpu()
pinn_infer_time = time.time() - pinn_infer_start

u_exact = u_exact_test

error = torch.norm(u_pred - u_exact) / torch.norm(u_exact)
linf_error_pinn = torch.max(torch.abs(u_pred - u_exact)).item()

print(f"PINN inference time : {pinn_infer_time:.4f}s")
print(f"PINN rel L2 error   : {error.item():.4e}")
print(f"PINN L-inf error    : {linf_error_pinn:.4e}")

u_pred_grid = u_pred.reshape(1000, 1000)
u_exact_grid = u_exact.reshape(1000, 1000)
u_abs_error = torch.abs(u_pred - u_exact).reshape(1000, 1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1 - exact solution
im1 = axes[0].pcolormesh(T.numpy(), X.numpy(), u_exact_grid.numpy(), cmap='hot')
axes[0].set_title('Exact Solution')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x')
plt.colorbar(im1, ax=axes[0])

# Plot 2 - PINN prediction
im2 = axes[1].pcolormesh(T.numpy(), X.numpy(), u_pred_grid.numpy(), cmap='hot')
axes[1].set_title('PINN Prediction')
axes[1].set_xlabel('t')
axes[1].set_ylabel('x')
plt.colorbar(im2, ax=axes[1])

# Plot 3 - absolute error
im3 = axes[2].pcolormesh(T.numpy(), X.numpy(), u_abs_error.numpy(), cmap='hot')
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('t')
axes[2].set_ylabel('x')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

**TODO:** Explain what this convergence study demonstrates about the Crank-Nicolson solver's accuracy as grid resolution increases.

In [ ]:
conv_errors = []
N_values = [10, 50, 100, 500, 1000, 5000]

for nx in N_values:
    x_c, t_c, u_c, conv_error = fd_solver(nx, nx)
    conv_errors.append(conv_error)


plt.figure()
plt.loglog(N_values, conv_errors, 'bo-', label='CN Error')
plt.xlabel('N_x')
plt.ylabel('Relative L2 Error')
plt.title('FD Convergence')
plt.grid(True)
plt.show()

**TODO:** Explain what the five-panel plot and the Baseline/Tuned/FD table below are comparing, and why FD's cost is measured on CPU while the PINN's is measured on GPU.

In [ ]:
fd_start = time.time()
x_fd, t_fd, u_fd, _ = fd_solver(1000, 1000, compute_error=False)
fd_time = time.time() - fd_start

X_fd, T_fd = np.meshgrid(x_fd, t_fd, indexing='ij')
u_exact_fd = np.sin(np.pi * X_fd) * np.exp(-forward_config["true_alpha"] * (np.pi**2) * T_fd)
error_fd = np.linalg.norm(u_fd - u_exact_fd) / np.linalg.norm(u_exact_fd)
linf_error_fd = np.max(np.abs(u_fd - u_exact_fd))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Exact
im1 = axes[0].pcolormesh(T.numpy(), X.numpy(), u_exact_grid.numpy(), cmap='hot')
axes[0].set_title('Exact')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x')
plt.colorbar(im1, ax=axes[0])

# PINN prediction
im2 = axes[1].pcolormesh(T.numpy(), X.numpy(), u_pred_grid.numpy(), cmap='hot')
axes[1].set_title('PINN')
axes[1].set_xlabel('t')
plt.colorbar(im2, ax=axes[1])

# FD prediction
im3 = axes[2].pcolormesh(T.numpy(), X.numpy(), u_fd, cmap='hot')
axes[2].set_title('FD (CN)')
axes[2].set_xlabel('t')
plt.colorbar(im3, ax=axes[2])

# PINN error
im4 = axes[3].pcolormesh(T.numpy(), X.numpy(), u_abs_error.numpy(), cmap='hot')
axes[3].set_title('PINN Error')
axes[3].set_xlabel('t')
plt.colorbar(im4, ax=axes[3])

# FD error
im5 = axes[4].pcolormesh(T.numpy(), X.numpy(), np.abs(u_fd - u_exact_fd), cmap='hot')
axes[4].set_title('FD Error')
axes[4].set_xlabel('t')
plt.colorbar(im5, ax=axes[4])

plt.tight_layout()
plt.show()

# Every Baseline/Tuned number below is mean ± std across the same 5
# seeds used in the retrain loops above, not a single lucky run's number. FD
# has no seed dependence (solving the same linear system from the same
# inputs is bit-for-bit deterministic), so it gets one value instead.
# FD's "Training time" is N/A because FD has no separate train/inference
# split -- fd_time already covers its whole one-shot solve cost, shown
# under "Inference time" instead. Baseline/FD's "Optuna search time" is
# N/A because only the Tuned PINN goes through hyperparameter search.
# (The winning Tuned model's single representative-run numbers -- used
# only to pick which model gets plotted above -- are printed separately
# near the plots, not repeated here.)
print(f"{'':25} {'Baseline':>14} {'Tuned':>14} {'FD (CN)':>14}")
print(f"{'-'*70}")
print(f"{'Training time (s)':25} {f'{baseline_train_time_mean:.2f}±{baseline_train_time_std:.2f}':>14} {f'{pinn_train_time_mean:.2f}±{pinn_train_time_std:.2f}':>14} {'N/A':>14}")
print(f"{'Inference time (s)':25} {f'{baseline_infer_time_mean:.4f}±{baseline_infer_time_std:.4f}':>14} {f'{pinn_infer_time_mean:.4f}±{pinn_infer_time_std:.4f}':>14} {fd_time:>14.4f}")
print(f"{'Rel L2 error':25} {f'{baseline_test_rel_l2_mean:.4e}±{baseline_test_rel_l2_std:.4e}':>14} {f'{pinn_test_rel_l2_mean:.4e}±{pinn_test_rel_l2_std:.4e}':>14} {error_fd:>14.4e}")
print(f"{'L-inf error':25} {f'{baseline_test_linf_mean:.4e}±{baseline_test_linf_std:.4e}':>14} {f'{pinn_test_linf_mean:.4e}±{pinn_test_linf_std:.4e}':>14} {linf_error_fd:>14.4e}")
print(f"{'Optuna search time (s)':25} {'N/A':>14} {optuna_forward_search_time:>14.2f} {'N/A':>14}")

**TODO:** (After the final run) Summarize what the final numbers show about tuning's benefit over baseline and how the PINN compares to FD on accuracy versus cost.

In [ ]:
# CPU control run: same frozen config, data, and seeds (0-4) as the GPU
# run above, just device="cpu". A hardware-context timing comparison, not
# a second accuracy claim.
cpu_forward_config = top_configs[best_config_idx]

cpu_forward_results = []
for seed in range(5):
    set_seed(seed)
    start = time.time()
    results = train_forward(cpu_forward_config, print_training=False, device=torch.device("cpu"))
    results['elapsed'] = time.time() - start
    cpu_forward_results.append(results)

cpu_forward_train_time_mean = float(np.mean([r['elapsed'] for r in cpu_forward_results]))
cpu_forward_train_time_std = float(np.std([r['elapsed'] for r in cpu_forward_results]))

# Field-evaluation time on CPU too, mirroring pinn_infer_time measured on
# GPU above. x_test_flat/t_test_flat are already CPU tensors (built without
# a device argument), so no .to(device) is needed here.
cpu_forward_infer_times = []
for r in cpu_forward_results:
    infer_start = time.time()
    with torch.no_grad():
        _ = r["model"](x_test_flat, t_test_flat)
    cpu_forward_infer_times.append(time.time() - infer_start)
cpu_forward_infer_time_mean = float(np.mean(cpu_forward_infer_times))
cpu_forward_infer_time_std = float(np.std(cpu_forward_infer_times))

print(f"\nCPU control (winning config, device='cpu', 5 seeds):")
print(f"  Training time:         {cpu_forward_train_time_mean:.2f} ± {cpu_forward_train_time_std:.2f} sec  (GPU: {pinn_train_time_mean:.2f} ± {pinn_train_time_std:.2f} sec)")
print(f"  Field-evaluation time: {cpu_forward_infer_time_mean:.4f} ± {cpu_forward_infer_time_std:.4f} sec  (GPU: {pinn_infer_time:.4f} sec)")

In [ ]:
# Sensitivity sweep: vary the forward problem's true diffusivity
# (true_alpha) across a few values, comparing the tuned PINN (the winning
# architecture/hyperparameters from the retrain loop above, frozen -- NOT
# re-tuned per value, for the same reason the inverse sweeps below freeze
# winning_inverse_config: this tests robustness of the already-selected
# config, not a repeat of the expensive HPO search) against FD at each.
# Tests whether the "is PINN-tuning worth it" conclusion holds generally,
# or is specific to the single true_alpha=0.4 value everything else in
# this notebook reports. The true_alpha=0.4 point reuses the already-
# computed final comparison results above instead of retraining a
# duplicate of the identical experiment. Same alpha values used in the
# inverse alpha sweep later, so the two problems' results are directly
# comparable. Seeds 20-24 used here, distinct from forward's baseline/
# retrain-loop seeds (0-4) purely for clarity when tracing through the
# notebook -- forward has no shared-dataset leakage risk the way inverse
# does (no external noisy observations to accidentally reuse), so this
# is a readability choice, not a correctness requirement.
ALPHA_SWEEP_VALUES = [0.1, 0.4, 0.7, 1.0]
winning_forward_config = top_configs[best_config_idx]
alpha_sweep_results = {}
alpha_sweep_start = time.time()

for alpha_val in ALPHA_SWEEP_VALUES:
    u_exact_alpha = torch.sin(torch.pi * x_test_flat) * torch.exp(-alpha_val * (torch.pi**2) * t_test_flat)

    if alpha_val == forward_config["true_alpha"]:
        alpha_sweep_results[alpha_val] = {
            "pinn_rel_l2_mean": pinn_test_rel_l2_mean,
            "pinn_rel_l2_std": pinn_test_rel_l2_std,
            "fd_rel_l2": error_fd,
        }
        print(f"true_alpha={alpha_val}: reused final comparison results (identical config)")
        continue

    sweep_forward_config = {**winning_forward_config, "true_alpha": alpha_val}
    pinn_errors = []
    for seed in range(20, 25):
        set_seed(seed)
        r = train_forward(sweep_forward_config, print_training=False)
        model_device = next(r["model"].parameters()).device
        with torch.no_grad():
            u_pred_alpha = r["model"](x_test_flat.to(model_device), t_test_flat.to(model_device)).cpu()
        pinn_errors.append((torch.norm(u_pred_alpha - u_exact_alpha) / torch.norm(u_exact_alpha)).item())

    _, _, u_fd_alpha, _ = fd_solver(1000, 1000, alpha=alpha_val, compute_error=False)
    X_fd_a, T_fd_a = np.meshgrid(x_fd, t_fd, indexing='ij')
    u_exact_fd_alpha = np.sin(np.pi * X_fd_a) * np.exp(-alpha_val * (np.pi**2) * T_fd_a)
    fd_error_alpha = np.linalg.norm(u_fd_alpha - u_exact_fd_alpha) / np.linalg.norm(u_exact_fd_alpha)

    alpha_sweep_results[alpha_val] = {
        "pinn_rel_l2_mean": float(np.mean(pinn_errors)),
        "pinn_rel_l2_std": float(np.std(pinn_errors)),
        "fd_rel_l2": float(fd_error_alpha),
    }
    print(f"true_alpha={alpha_val}: PINN rel L2 {np.mean(pinn_errors):.4e} ± {np.std(pinn_errors):.4e} | "
          f"FD rel L2 {fd_error_alpha:.4e}")

alpha_sweep_time = time.time() - alpha_sweep_start
print(f"\nForward alpha sensitivity sweep total time: {alpha_sweep_time:.2f}s")

plt.figure(figsize=(7, 5))
xs = sorted(alpha_sweep_results.keys())
pinn_means = [alpha_sweep_results[x]["pinn_rel_l2_mean"] for x in xs]
pinn_stds = [alpha_sweep_results[x]["pinn_rel_l2_std"] for x in xs]
fd_vals = [alpha_sweep_results[x]["fd_rel_l2"] for x in xs]
plt.errorbar(xs, pinn_means, yerr=pinn_stds, marker='o', label='Tuned PINN', capsize=4)
plt.plot(xs, fd_vals, marker='s', label='FD (CN)')
plt.yscale('log')
plt.xlabel('True diffusivity (alpha)')
plt.ylabel('Rel L2 error')
plt.title('Rel L2 Error vs. True Diffusivity')
plt.legend()
plt.grid(True)
plt.show()

**TODO:** (After the final run) Summarize whether the tuning-vs-FD accuracy gap holds across all tested true_alpha values or only near the default.